In [1]:
'''
Objective: This code initializes required libraries for the Carla simulator to add a vehicle, and attach LiDAR and depth sensors to it. The libraries facilitate the simulation and visualization of the vehicle and sensor data.

Libraries Overview:
    - carla: Provides the interface to interact with the Carla simulator, allowing vehicle and sensor manipulation.
    - math: Supplies mathematical functions (e.g., for positioning, rotations).
    - random: Generates random numbers, useful for randomizing vehicle or sensor placement.
    - time: Provides time control (e.g., for pauses and timing).
    - numpy (np): Enables numerical operations on arrays, important for sensor data processing.
    - cv2: Handles image processing and display (for camera sensor data).
    - matplotlib and cm: Used for color mapping and data visualization (e.g., for depth sensor).
'''
import carla 
import math
import random
import time
import numpy as np
import cv2
from matplotlib import cm
import matplotlib

In [2]:
'''
Objective: Establish a connection to the Carla simulator and access the simulation world.

1. carla.Client('localhost', 2000):
    - Purpose: Creates a client to connect to the Carla simulator running on localhost at port 2000.
    - Usage: Acts as the main interface to communicate with the Carla server.

2. client.get_world():
    - Purpose: Retrieves the current simulation environment or "world" in Carla.
    - Usage: Provides access to all elements in the simulation, such as vehicles, sensors, and actors, allowing manipulation within the world.
'''
client = carla.Client('localhost', 2000)
world = client.get_world()

In [3]:
'''
Objective: Retrieve available spawn points within the Carla simulation world.

1.world.get_map():
    - Purpose: Fetches the current map of the simulation world.
    - Usage: Provides information related to the simulation environment's layout (e.g., roads, intersections).

2.get_spawn_points():
    - Purpose: Returns a list of predefined spawn points (positions and orientations) on the map where vehicles can be placed.
    - Usage: Used to randomly or specifically select a location to spawn vehicles.
'''
spawn_points = world.get_map().get_spawn_points()

In [4]:
'''
Objective: Access the blueprint library to retrieve vehicle and sensor definitions.

1. world.get_blueprint_library():
    - Purpose: Fetches the blueprint library, which contains definitions for various actors, including vehicles, sensors, and props.
    - Usage: Enables selection and instantiation of specific vehicle or sensor types within the simulation, allowing customization and flexibility in vehicle configurations.
'''
bp_lib = world.get_blueprint_library()

In [5]:
'''
Objective: Spawn a specific vehicle in the Carla simulation.

1. `bp_lib.find('vehicle.tesla.cybertruck')`:
   - Purpose: Locates the blueprint for the Tesla Cybertruck within the blueprint library.
   - Usage: Retrieves the vehicle's specifications to allow for instantiation in the simulation.

2. `world.try_spawn_actor(vehicle_bp, spawn_points[0])`:
   - Purpose: Attempts to spawn an actor (the Tesla Cybertruck) at the first available spawn point.
   - Usage: Creates an instance of the vehicle in the simulation; if spawning is successful, it returns the vehicle object, allowing further interaction.
'''

vehicle_bp = bp_lib.find('vehicle.tesla.cybertruck')
vehicle = world.try_spawn_actor(vehicle_bp, spawn_points[0])

In [14]:
vehicle.set_autopilot(True)

In [6]:
'''
Objective: Define the initial camera position for the simulation.

1. `CAMERA_POS_Z = 3`:
   - Purpose: Sets the Z-axis position of the camera above the vehicle.
   - Usage: Specifies how high the camera will be placed relative to the vehicle, enhancing the field of view.

2. `CAMERA_POS_X = 0`:
   - Purpose: Sets the X-axis position of the camera.
   - Usage: Defines the lateral position of the camera relative to the vehicle, allowing for adjustments in viewpoint.

3. `camera_init_trans = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))`:
   - Purpose: Creates a transformation object representing the camera's initial position.
   - Usage: Combines the defined X and Z positions into a transform that can be used when attaching the camera to the vehicle.
'''
CAMERA_POS_Z = 3
CAMERA_POS_X = 0
camera_init_trans = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))

In [7]:
camera_bp = bp_lib.find('sensor.camera.rgb')
camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=vehicle)

In [8]:
def rgb_callback(image, data_dict):
    data_dict['rgb_image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [9]:
sem_cam_bp = bp_lib.find('sensor.camera.instance_segmentation')
sem_cam = world.spawn_actor(sem_cam_bp, camera_init_trans, attach_to = vehicle)

In [10]:
def sem_callback(image, data_dict):
    image.convert(carla.ColorConverter.CityScapesPalette)
    data_dict['sem_image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [11]:
image_w = camera_bp.get_attribute("image_size_x").as_int()
image_h = camera_bp.get_attribute("image_size_y").as_int()

In [12]:
sensor_data = {'rgb_image': np.zeros((image_h, image_w, 4)),
               'sem_image': np.zeros((image_h, image_w, 4))}

In [13]:
camera.listen(lambda image: rgb_callback(image, sensor_data))
sem_cam.listen(lambda image: sem_callback(image, sensor_data))

while True:
    #top_row = np.concatenate((sensor_data['rgb_image'], sensor_data['sem_image']), axis=1)
    #lower_row = np.concatenate((sensor_data['rgb_image'], sensor_data['sem_image']), axis=1)
    #tiled = np.concatenate((top_row, lower_row), axis=0)

    cv2.imshow('All cameras', sensor_data['sem_image'])

    if cv2.waitKey(1) == ord('q'):
        break
        
camera.stop()
sem_cam.stop()
cv2.destroyAllWindows()